## Analysis extension #4 

Paired clean joint versus development-only MLM control

AE1b now generates **all representation training**. This notebook is analysis-only.

For each paired seed, it loads:

- the clean joint MLM trained on development + Burdziak;
- the clean development-**only** MLM trained on the same developmental corpus and preprocessing but without Burdziak training cells.

It then:
 1)rebuilds the developmental PCA independently inside each encoder, 
 2)applies the finalized AE3 biological direction specification, 
 3)quantifies Burdziak developmental reconstruction, 
 4)fits residual PCA on K-series cells only, 
 5)projects N1/N2 afterward, and 
 6)compares the results by paired seed.

This is a direct analysis about the concern that tumor cells influenced the original representation before the developmental reference was constructed.

(*note that in the notebook last running AE was labeled WP, work_package*)


## 0. Load the AE1b manifest and audited aligned datasets


In [ ]:
from pathlib import Path
import os, json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import spearmanr

from sklearn.decomposition import PCA

import torch
from transformers import BertForMaskedLM

DATA_ROOT = Path("/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data")
AUDITED_ROOT = DATA_ROOT / "revision_audited_mlm"
MANIFEST_PATH = AUDITED_ROOT / "manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"{MANIFEST_PATH}\nRun AE1b audited retraining first."
    )

with open(MANIFEST_PATH) as f:
    MANIFEST = json.load(f)

SEEDS = [int(s) for s in MANIFEST["seeds"]]
DEV_H5AD = Path(MANIFEST["dev_h5ad"])
CANCER_H5AD = Path(MANIFEST["cancer_h5ad"])

OUTDIR = AUDITED_ROOT / "analysis" / "ae4_joint_vs_development_only"
OUTDIR.mkdir(parents=True, exist_ok=True)

K_DEV = 75
N_RESIDUAL_PCS_TO_ANNOTATE = 10
NORMAL_CONDITIONS = {"N1", "N2"}
CANCER_STATE_COL = "cancer_state"
SAMPLE_COL = "batch" #This makes AE4 use the same sample definition as final AE3.


MALIGNANT_TRANSITIONS = [
    ("early", "ductal", "transitional"),
    ("late", "transitional", "malignant_ductal"),
    ("total", "ductal", "malignant_ductal"),
]

A_dev = sc.read_h5ad(DEV_H5AD)
A_can = sc.read_h5ad(CANCER_H5AD)

print("Seeds:", SEEDS)
print("Development:", A_dev.shape)
print("Burdziak:", A_can.shape)
print("Outputs:", OUTDIR)


Seeds: [17, 42]
Development: (34760, 15376)
Burdziak: (28131, 15376)
Outputs: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp4_joint_vs_development_only


## 1. Load exact AE1b embeddings

- The saved evaluation corpus preserves biological provenance in `domain.npy` (`0=development`, `1=Burdziak`). 
- In the development-only model the Burdziak query sequences themselves use the learned `[DEV]` token; this rule is recorded in each model's metadata.

In [ ]:
def load_encoder(scope, seed):
    model_dir = Path(MANIFEST["models"][scope][str(seed)])

    with open(model_dir / "vocab_meta.json") as f:
        meta = json.load(f)

    # Definitive CLS embeddings written by AE1b.
    dev_cache = model_dir / "X_dev_cls.npy"
    can_cache = model_dir / "X_cancer_cls.npy"

    if dev_cache.exists() and can_cache.exists():
        Xdev = np.load(dev_cache)
        Xcan = np.load(can_cache)
    else:
        model = BertForMaskedLM.from_pretrained(model_dir)
        tok = np.load(model_dir / "X_tokens.npy", mmap_mode="r")
        attn = np.load(model_dir / "X_attn.npy", mmap_mode="r")
        dom = np.load(model_dir / "domain.npy")

        @torch.no_grad()
        def embed(T, A, batch_size=32):
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model.to(device)
            model.eval()
            chunks = []

            for i in range(0, len(T), batch_size):
                x = torch.as_tensor(
                    T[i:i+batch_size],
                    dtype=torch.long,
                    device=device,
                )
                a = torch.as_tensor(
                    A[i:i+batch_size],
                    dtype=torch.long,
                    device=device,
                )
                out = model.bert(
                    input_ids=x,
                    attention_mask=a,
                    return_dict=True,
                )
                chunks.append(out.last_hidden_state[:, 1, :].cpu())

            return torch.cat(chunks, dim=0).numpy()

        E = embed(tok, attn)
        Xdev = E[dom == 0]
        Xcan = E[dom == 1]

    # Dimension checks.
    if Xdev.shape[0] != A_dev.n_obs or Xcan.shape[0] != A_can.n_obs:
        raise RuntimeError(
            f"{scope} seed {seed}: embedding/cell-count mismatch"
        )

    # Exact row-identity/order checks.
    dev_names = np.load(
        model_dir / "dev_obs_names.npy",
        allow_pickle=True,
    ).astype(str)

    can_names = np.load(
        model_dir / "cancer_obs_names.npy",
        allow_pickle=True,
    ).astype(str)

    assert np.array_equal(
        dev_names,
        A_dev.obs_names.astype(str).to_numpy(),
    ), f"{scope} seed {seed}: developmental row order mismatch"

    assert np.array_equal(
        can_names,
        A_can.obs_names.astype(str).to_numpy(),
    ), f"{scope} seed {seed}: cancer row order mismatch"

    return {
        "scope": scope,
        "seed": seed,
        "model_dir": model_dir,
        "meta": meta,
        "Xdev": Xdev,
        "Xcan": Xcan,
    }

encoders = {
    (scope, seed): load_encoder(scope, seed)
    for seed in SEEDS
    for scope in ["joint", "development_only"]
}

print("Loaded", len(encoders), "audited encoders with row-order checks passed.")


Loaded 4 audited encoders with row-order checks passed.


## 2. Geometry and finalized WP3 developmental directions

The **biological definitions are fixed across model families**; the numerical vectors are reconstructed independently in each representation.

No old `foregut_other` axis is used.


In [ ]:
def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def fit_developmental_reference(X_dev, k=K_DEV):
    k_eff = min(k, X_dev.shape[1], X_dev.shape[0] - 1)
    mu = X_dev.mean(axis=0)
    Xd = X_dev - mu
    pca = PCA(n_components=k_eff, svd_solver="full").fit(Xd)
    coords = pca.transform(Xd)
    basis = pca.components_.T
    return mu, pca, basis, coords

def project_to_reference(X, mu, pca, basis):
    centered = X - mu
    coords = pca.transform(centered)
    recon = coords @ basis.T
    residual = centered - recon
    proj2 = np.sum(recon**2, axis=1)
    resid2 = np.sum(residual**2, axis=1)
    return {
        "coords": coords,
        "residual": residual,
        "projection_score": proj2 / (proj2 + resid2 + 1e-12),
        "residual_norm": np.sqrt(resid2),
        "global_fraction": float(proj2.sum() / (proj2.sum() + resid2.sum() + 1e-12)),
    }

def ae3_state_masks(obs):
    ct = obs["celltype_update"].astype(str)
    day = obs["dev_day"].astype(float)
    return {
        "MPC_like": ((ct == "Pancreatic acinar cells") & (day <= 11.5)).to_numpy(),
        "tip_acinar": ((ct == "Pancreatic acinar cells") & (day >= 13.75) & (day <= 16.5)).to_numpy(),
        "endocrine_secondary": ((ct == "Pancreatic islets") & (day >= 13.75) & (day <= 16.5)).to_numpy(),
        "endocrine_late": ((ct == "Pancreatic islets") & (day >= 16.75)).to_numpy(),
        "biliary_early": ((ct == "Biliary epithelial cells") & (day >= 11.75) & (day <= 13.5)).to_numpy(),
        "biliary_secondary": ((ct == "Biliary epithelial cells") & (day >= 13.75) & (day <= 16.5)).to_numpy(),
    }

def equal_day_centroid(X, obs, mask, min_day_n=3):
    day = obs["dev_day"].astype(float).to_numpy()
    cents = []
    for d in sorted(pd.unique(obs.loc[mask, "dev_day"].dropna())):
        m = np.asarray(mask, dtype=bool) & (day == float(d))
        if int(m.sum()) >= min_day_n:
            cents.append(np.asarray(X)[m].mean(axis=0))
    if not cents:
        raise ValueError("No eligible developmental days.")
    return np.vstack(cents).mean(axis=0)

def build_ae3_directions(dev_pc, obs):
    masks = ae3_state_masks(obs)
    counts = {k: int(v.sum()) for k, v in masks.items()}
    if min(counts.values()) < 30:
        raise RuntimeError(f"Unexpectedly small AE3 state: {counts}")

    C = {name: equal_day_centroid(dev_pc, obs, mask) for name, mask in masks.items()}
    specs = {
        "MPC_to_tip_acinar": ("MPC_like", "tip_acinar"),
        # Endpoint contrast, not a literal lineage edge.
        "MPC_to_endocrine_islet": ("MPC_like", "endocrine_secondary"),
        "endocrine_maturation": ("endocrine_secondary", "endocrine_late"),
        "biliary_maturation": ("biliary_early", "biliary_secondary"),
    }
    return {
        name: C[end] - C[start]
        for name, (start, end) in specs.items()
    }


def sample_equal_state_centroid(
    coords,
    obs,
    state,
    state_col=CANCER_STATE_COL,
    sample_col=SAMPLE_COL,
):
    coords = np.asarray(coords)

    states = obs[state_col].astype(str).to_numpy()
    samples = obs[sample_col].astype(str).to_numpy()

    mask = states == str(state)
    sample_ids = pd.unique(samples[mask])

    if len(sample_ids) == 0:
        raise ValueError(f"No biological samples found for state {state!r}")

    sample_centroids = []

    for sample in sample_ids:
        smask = mask & (samples == sample)

        if not np.any(smask):
            continue

        sample_centroids.append(
            coords[smask].mean(axis=0)
        )

    if not sample_centroids:
        raise ValueError(
            f"No valid sample centroids found for state {state!r}"
        )

    return np.mean(
        np.stack(sample_centroids, axis=0),
        axis=0,
    )


def malignant_alignment(coords, obs, dev_dirs):
    rows = []

    # Equal-weight each biological sample within each malignant state.
    state_centroids = {
        state: sample_equal_state_centroid(coords, obs, state)
        for state in ["ductal", "transitional", "malignant_ductal"]
    }

    for transition, start, end in MALIGNANT_TRANSITIONS:
        v = state_centroids[end] - state_centroids[start]

        for dname, dvec in dev_dirs.items():
            rows.append({
                "transition": transition,
                "developmental_direction": dname,
                "cosine_similarity": cosine(v, dvec),
            })

    return pd.DataFrame(rows)



## 3. Curated malignant residual programs

Residual PCA is fitted on K-series cells only; N1/N2 never define these residual axes.


In [4]:
CANCER_PROGRAMS_MOUSE = {
    "emt_invasion": ["Vim","Fn1","Zeb1","Zeb2","Snai1","Snai2","Twist1","Cdh2","Itga5","Itgav","Itgb1","Mmp2","Mmp9","Mmp14","Col1a1","Col1a2","Col3a1","Sparc","Tnc","Lox","Postn"],
    "epithelial_identity": ["Epcam","Cdh1","Krt8","Krt18","Krt19","Krt7","Cldn3","Cldn4","Cldn7","Ocln","Tjp1","Muc1"],
    "inflammatory_signaling": ["Nfkb1","Nfkb2","Rela","Relb","Tnf","Tnfaip3","Il1a","Il1b","Il6","Cxcl1","Cxcl2","Cxcl10","Ccl2","Ccl5","Ptgs2","Socs3","Icam1"],
    "interferon_response": ["Stat1","Stat2","Irf1","Irf7","Irf9","Isg15","Ifit1","Ifit2","Ifit3","Ifitm1","Ifitm2","Ifitm3","Oas1a","Oas1g","Oas2","Oas3","Mx1","Rsad2","Usp18","Dhx58"],
    "hypoxia": ["Hif1a","Vegfa","Slc2a1","Ldha","Pdk1","Eno1","Aldoa","Bnip3","Ndrg1","Adm","Car9","Serpine1","Ero1a","Ankrd37"],
    "oxidative_stress": ["Nfe2l2","Keap1","Hmox1","Nqo1","Gclc","Gclm","Gsr","Gpx2","Gpx4","Prdx1","Txn1","Txnrd1","Srxn1","Sqstm1","Ftl1","Fth1"],
    "unfolded_protein_response": ["Hspa5","Hsp90b1","Ddit3","Atf3","Atf4","Atf6","Xbp1","Ern1","Eif2ak3","Pdia3","Pdia4","Pdia6","Dnajb9","Herpud1","Derl3","Manf","Creld2"],
    "kras_mapk_signaling": ["Dusp4","Dusp5","Dusp6","Etv1","Etv4","Etv5","Spry1","Spry2","Spry4","Fos","Fosl1","Jun","Junb","Egr1","Egr2","Ets1","Myc","Mapk1","Mapk3"],
    "myc_targets": ["Myc","Max","Mxd1","Npm1","Ncl","Odc1","Cad","Nme1","Nme2","Hspd1","Hspe1","Tfrc","Slc2a1","Phb","Pa2g4","Hnrnpa1","Hnrnpa2b1","Eif4a1"],
    "p53_pathway": ["Trp53","Cdkn1a","Mdm2","Bax","Bbc3","Pmaip1","Gadd45a","Gadd45g","Sesn1","Sesn2","Ddb2","Phlda3","Zmat3","Rrm2b","Ccng1"],
    "cell_cycle": ["Mki67","Top2a","Pcna","Mcm2","Mcm3","Mcm4","Mcm5","Mcm6","Mcm7","Cdk1","Ccnb1","Ccnb2","Cenpa","Cenpe","Cenpf","Tpx2","Bub1b","Ube2c","Aspm","Nusap1","Kif11","Kif20b"],
    "dna_damage_repair": ["Brca1","Brca2","Rad51","Rad51ap1","Msh2","Msh6","Pms2","Pcna","Rpa1","Rpa2","Chek1","Chek2","Atm","Atr","Fanca","Fancc","Xrcc5","Xrcc6"],
    "glycolysis_metabolic_rewiring": ["Slc2a1","Hk1","Hk2","Pfkl","Pfkp","Aldoa","Gapdh","Pgam1","Eno1","Pkm","Ldha","Pdk1","Slc16a3"],
    "oxidative_phosphorylation_mitochondrial": ["Ndufa2","Ndufa4","Ndufb3","Ndufb5","Sdha","Sdhb","Uqcrc1","Uqcrc2","Cyc1","Cox4i1","Cox5a","Atp5f1a","Atp5f1b","Atp5mc1"],
    "ecm_remodeling_invasion": ["Fn1","Col1a1","Col1a2","Col3a1","Col4a1","Col4a2","Col5a1","Col5a2","Sparc","Tnc","Postn","Lox","Loxl2","Mmp2","Mmp9","Mmp14","Itga5","Itgav","Itgb1","Vcan","Thbs1"],
    "antigen_presentation_mhc_i": ["B2m","H2-K1","H2-D1","Tap1","Tap2","Tapbp","Psmb8","Psmb9","Psmb10","Nlrc5"],
    "antigen_presentation_mhc_ii": ["H2-Aa","H2-Ab1","H2-Eb1","Cd74","Ciita","H2-DMa","H2-DMb1","H2-Ob"],
    "stemness_progenitor_like": ["Sox9","Prom1","Aldh1a1","Aldh1a3","Lgr5","Krt19","Epcam","Tacstd2","Muc1","Nes","Vim","Lgals1","S100a10"],
    "therapy_resistance_stress_tolerance": ["Aldh1a1","Aldh1a3","Abcb1a","Abcb1b","Abcc1","Abcc3","Abcg2","Bcl2","Bcl2l1","Mcl1","Hspa1a","Hspa1b","Hspa5","Hmox1","Nqo1","Sod2","Gpx2","Lgals1","Tgfb1","Vim"],
    "tgfb_signaling": ["Tgfb1","Tgfb2","Tgfbr1","Tgfbr2","Smad2","Smad3","Smad4","Serpine1","Id1","Id2","Id3","Skil","Snai1","Snai2"],
    "apoptosis": ["Bax","Bak1","Bbc3","Pmaip1","Casp3","Casp7","Casp8","Casp9","Apaf1","Bid","Bad","Bcl2","Bcl2l1","Mcl1"],
    "senescence_sasp": ["Cdkn1a","Cdkn2a","Trp53","Il6","Cxcl1","Cxcl2","Ccl2","Serpine1","Mmp3","Mmp9","Igfbp3","Igfbp5","Gadd45a"],
    "ap1_immediate_early_response": ["Fos","Fosb","Fosl1","Fosl2","Jun","Junb","Jund","Atf3","Egr1","Egr2","Dusp1","Dusp5","Dusp6","Ier2","Csrnp1","Zfp36","Btg2","Ccn1","Klf6"],
}

def score_programs_log1p_cpm(
    adata,
    programs,
    target_sum=1e4,
    min_genes=5,
    gene_symbol_col="symbol",
):
    """
    Mean log1p library-normalized expression across present
    program genes, matching the WP2 definition.
    """
    X = (
        adata.X.tocsr()
        if sp.issparse(adata.X)
        else np.asarray(adata.X)
    )

    if gene_symbol_col in adata.var.columns:
        symbols = (
            adata.var[gene_symbol_col]
            .astype(str)
            .to_numpy()
        )
    else:
        symbols = adata.var_names.astype(str).to_numpy()

    gene_to_idx = {}
    for i, g in enumerate(symbols):
        gene_to_idx.setdefault(g, i)

    if sp.issparse(X):
        totals = np.asarray(X.sum(axis=1)).ravel()
    else:
        totals = X.sum(axis=1)

    scale = target_sum / np.maximum(totals, 1.0)

    out = {}
    coverage = []

    for pname, genes in programs.items():
        present = [
            g for g in genes
            if g in gene_to_idx
        ]

        coverage.append({
            "program": pname,
            "n_total": len(genes),
            "n_present": len(present),
            "fraction_present": (
                len(present) / max(len(genes), 1)
            ),
        })

        if len(present) < min_genes:
            continue

        idx = [
            gene_to_idx[g]
            for g in present
        ]

        if sp.issparse(X):
            Xi = X[:, idx].toarray()
        else:
            Xi = X[:, idx]

        Xi = np.log1p(
            Xi * scale[:, None]
        )

        out[pname] = Xi.mean(axis=1)

    return (
        pd.DataFrame(out, index=adata.obs_names),
        pd.DataFrame(coverage),
    )


def correlate_residual_programs(residual_coords, program_scores, n_pcs=N_RESIDUAL_PCS_TO_ANNOTATE):
    rows = []
    n_pcs = min(n_pcs, residual_coords.shape[1])
    for j in range(n_pcs):
        axis = residual_coords[:, j]
        for program in program_scores.columns:
            r = spearmanr(axis, program_scores[program], nan_policy="omit").correlation
            rows.append({"axis": f"ResCLS_PC{j+1}", "program": program, "spearman_r": r, "abs_spearman_r": abs(r)})
    return pd.DataFrame(rows).sort_values(["axis", "abs_spearman_r"], ascending=[True, False])


In [5]:
cancer_program_scores, program_coverage = score_programs_log1p_cpm(A_can,CANCER_PROGRAMS_MOUSE,min_genes=5)

program_coverage.to_csv(OUTDIR / "program_coverage_log1pCPM.csv",index=False)
cancer_program_scores.to_csv(OUTDIR / "cancer_program_scores_log1pCPM.csv")


## 4. Analyze every paired encoder

In [ ]:
summary_rows = []
alignment_frames = []
state_frames = []
program_frames = []

states = A_can.obs[CANCER_STATE_COL].astype(str).to_numpy()
conditions = A_can.obs["condition"].astype(str).to_numpy()
kseries_mask = np.char.startswith(conditions.astype(str), "K")
normal_mask = np.isin(conditions, list(NORMAL_CONDITIONS))

for (scope, seed), E in encoders.items():
    print("\n", "="*70)
    print(scope, "seed", seed)
    print("="*70)

    mu, pca_dev, basis, dev_pc = fit_developmental_reference(E["Xdev"], K_DEV)
    dev_proj = project_to_reference(E["Xdev"], mu, pca_dev, basis)
    can_proj = project_to_reference(E["Xcan"], mu, pca_dev, basis)

    dev_dirs = build_ae3_directions(dev_pc, A_dev.obs)
    
    align = malignant_alignment(can_proj["coords"],A_can.obs,dev_dirs)
    align["scope"] = scope
    align["seed"] = seed
    alignment_frames.append(align)

    # K-series-only residual PCA.
    R = can_proj["residual"]
    rmean = R[kseries_mask].mean(axis=0)
    nres = min(R.shape[1], int(kseries_mask.sum()) - 1)
    p_res = PCA(n_components=nres, svd_solver="full").fit(R[kseries_mask] - rmean)
    Rcoords = p_res.transform(R - rmean)

    pcorr = correlate_residual_programs(
        Rcoords[kseries_mask],
        cancer_program_scores.iloc[np.flatnonzero(kseries_mask)].reset_index(drop=True),
        n_pcs=N_RESIDUAL_PCS_TO_ANNOTATE,
    )
    pcorr["scope"] = scope
    pcorr["seed"] = seed
    program_frames.append(pcorr)

    cell = pd.DataFrame({
        "condition": conditions,
        "state": states,
        "projection_score": can_proj["projection_score"],
        "residual_norm": can_proj["residual_norm"],
    })
    st = (
        cell.groupby(["condition", "state"], as_index=False)
        .agg(
            n=("projection_score", "size"),
            projection_median=("projection_score", "median"),
            projection_mean=("projection_score", "mean"),
            residual_norm_median=("residual_norm", "median"),
        )
    )
    st["scope"] = scope
    st["seed"] = seed
    state_frames.append(st)

    rv = np.cumsum(p_res.explained_variance_ratio_)
    row = {
        "scope": scope,
        "seed": seed,
        "dev_variance_at_K": float(pca_dev.explained_variance_ratio_.sum()),
        "dev_projection_median": float(np.median(dev_proj["projection_score"])),
        "cancer_projection_global_fraction": can_proj["global_fraction"],
        "cancer_projection_median": float(np.median(can_proj["projection_score"])),
        "normal_projection_median": float(np.median(can_proj["projection_score"][normal_mask])),
        "Kseries_projection_median": float(np.median(can_proj["projection_score"][kseries_mask])),
        "residual_top10_variance": float(rv[min(9, len(rv)-1)]),
        "query_domain_rule": E["meta"].get("query_domain_rule", ""),
    }
    for _, a in align.iterrows():
        row[f"cos__{a['transition']}__{a['developmental_direction']}"] = float(a["cosine_similarity"])
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
alignments = pd.concat(alignment_frames, ignore_index=True)
state_summary = pd.concat(state_frames, ignore_index=True)
program_corr = pd.concat(program_frames, ignore_index=True)

summary.to_csv(OUTDIR / "ae4_encoder_summary.csv", index=False)
alignments.to_csv(OUTDIR / "ae4_revised_direction_alignment.csv", index=False)
state_summary.to_csv(OUTDIR / "ae4_condition_state_projection.csv", index=False)
program_corr.to_csv(OUTDIR / "ae4_Kseries_residual_program_correlations.csv", index=False)

display(summary)



joint seed 17

development_only seed 17

joint seed 42

development_only seed 42


,scope,seed,dev_variance_at_K,dev_projection_median,cancer_projection_global_fraction,cancer_projection_median,normal_projection_median,Kseries_projection_median,residual_top10_variance,query_domain_rule,...,cos__early__endocrine_maturation,cos__early__biliary_maturation,cos__late__MPC_to_tip_acinar,cos__late__MPC_to_endocrine_islet,cos__late__endocrine_maturation,cos__late__biliary_maturation,cos__total__MPC_to_tip_acinar,cos__total__MPC_to_endocrine_islet,cos__total__endocrine_maturation,cos__total__biliary_maturation
0,joint,17,0.979956,0.980460,0.546546,0.545879,0.573475,0.540721,0.667350,DEV_for_development__CANCER_for_burdziak,...,-0.196743,0.243136,-0.136535,-0.016074,0.010318,-0.109964,-0.380081,0.051804,-0.179371,0.163298
1,development_only,17,0.977450,0.975444,0.952344,0.950206,0.962256,0.949034,0.497182,DEV_for_all_queries,...,-0.284917,0.133773,-0.312372,-0.457397,-0.112861,-0.216575,-0.456297,-0.218018,-0.265493,0.079328
2,joint,42,0.980711,0.982085,0.466932,0.465911,0.515986,0.452379,0.661753,DEV_for_development__CANCER_for_burdziak,...,-0.192868,0.180194,-0.002905,-0.117104,0.125986,-0.085511,-0.329938,0.031494,-0.113731,0.124616
3,development_only,42,0.976708,0.976520,0.957809,0.958164,0.960001,0.957891,0.517262,DEV_for_all_queries,...,-0.209159,0.293029,-0.454321,-0.422349,-0.111028,-0.068230,-0.571790,-0.198875,-0.200784,0.242664


In [7]:
# ============================================================
# Joint-model query-domain-token sensitivity
# Compare the same trained JOINT encoder with Burdziak queried
# using [CANCER] versus [DEV].
# ============================================================

@torch.no_grad()
def embed_token_matrix(
    model_dir,
    tokens,
    attention_mask,
    batch_size=64,
):
    model = BertForMaskedLM.from_pretrained(model_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    chunks = []

    for i in range(0, len(tokens), batch_size):
        x = torch.as_tensor(
            tokens[i:i+batch_size],
            dtype=torch.long,
            device=device,
        )
        a = torch.as_tensor(
            attention_mask[i:i+batch_size],
            dtype=torch.long,
            device=device,
        )

        out = model.bert(
            input_ids=x,
            attention_mask=a,
            return_dict=True,
        )

        # [DOMAIN] [CLS] ... -> CLS is position 1.
        chunks.append(
            out.last_hidden_state[:, 1, :].cpu()
        )

    result = torch.cat(chunks, dim=0).numpy()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


token_sensitivity_rows = []

for seed in SEEDS:
    E = encoders[("joint", seed)]
    model_dir = E["model_dir"]
    meta = E["meta"]

    tokens = np.load(
        model_dir / "X_tokens.npy",
        mmap_mode="r",
    )
    attn = np.load(
        model_dir / "X_attn.npy",
        mmap_mode="r",
    )
    domain = np.load(
        model_dir / "domain.npy"
    )

    cancer_rows = domain == 1

    cancer_tokens = np.asarray(
        tokens[cancer_rows]
    ).copy()

    cancer_attn = np.asarray(
        attn[cancer_rows]
    )

    DEV_DOM_ID = int(meta["DEV_DOM_ID"])
    CANCER_DOM_ID = int(meta["CANCER_DOM_ID"])

    # Sanity check: the saved joint query uses [CANCER].
    assert np.all(
        cancer_tokens[:, 0] == CANCER_DOM_ID
    )

    # Same cells, same trained joint model, but replace only
    # the leading query-domain token with [DEV].
    cancer_tokens_dev = cancer_tokens.copy()
    cancer_tokens_dev[:, 0] = DEV_DOM_ID

    Xcan_devtoken = embed_token_matrix(
        model_dir,
        cancer_tokens_dev,
        cancer_attn,
    )

    assert Xcan_devtoken.shape == E["Xcan"].shape

    # Fit the SAME developmental reference from the joint model.
    mu, pca_dev, basis, _ = fit_developmental_reference(
        E["Xdev"],
        K_DEV,
    )

    proj_cancer_token = project_to_reference(
        E["Xcan"],
        mu,
        pca_dev,
        basis,
    )

    proj_dev_token = project_to_reference(
        Xcan_devtoken,
        mu,
        pca_dev,
        basis,
    )

    rho = spearmanr(
        proj_cancer_token["projection_score"],
        proj_dev_token["projection_score"],
    ).correlation

    token_sensitivity_rows.append({
        "seed": seed,

        "CANCER_token_global_fraction":
            proj_cancer_token["global_fraction"],

        "DEV_token_global_fraction":
            proj_dev_token["global_fraction"],

        "DEV_minus_CANCER_global_fraction":
            proj_dev_token["global_fraction"]
            - proj_cancer_token["global_fraction"],

        "CANCER_token_projection_median":
            float(np.median(
                proj_cancer_token["projection_score"]
            )),

        "DEV_token_projection_median":
            float(np.median(
                proj_dev_token["projection_score"]
            )),

        "DEV_minus_CANCER_projection_median":
            float(
                np.median(proj_dev_token["projection_score"])
                - np.median(
                    proj_cancer_token["projection_score"]
                )
            ),

        "projection_score_spearman":
            float(rho),
    })


joint_domain_token_sensitivity = pd.DataFrame(
    token_sensitivity_rows
)

joint_domain_token_sensitivity.to_csv(
    OUTDIR / "wp4_joint_domain_token_sensitivity.csv",
    index=False,
)

display(joint_domain_token_sensitivity)

,seed,CANCER_token_global_fraction,DEV_token_global_fraction,DEV_minus_CANCER_global_fraction,CANCER_token_projection_median,DEV_token_projection_median,DEV_minus_CANCER_projection_median,projection_score_spearman
0,17,0.546546,0.540838,-0.005708,0.545879,0.538910,-0.006969,0.992711
1,42,0.466932,0.468796,0.001864,0.465911,0.468236,0.002325,0.997073


## 5. Paired seed comparison

The useful comparison is within seed: joint and development-only share preprocessing and initialization seed, while differing in whether Burdziak cells contributed to training.

Do *not* demand identical coordinates. Compare biological summaries: reconstruction ordering, revised developmental-direction alignment and residual organization.


In [ ]:
paired_rows = []

for seed in SEEDS:
    j = summary[(summary["seed"] == seed) & (summary["scope"] == "joint")].iloc[0]
    d = summary[(summary["seed"] == seed) & (summary["scope"] == "development_only")].iloc[0]

    row = {
        "seed": seed,
        "joint_cancer_projection": j["cancer_projection_global_fraction"],
        "devonly_cancer_projection": d["cancer_projection_global_fraction"],
        "devonly_minus_joint_cancer_projection": (
            d["cancer_projection_global_fraction"] - j["cancer_projection_global_fraction"]
        ),
        "joint_normal_projection_median": j["normal_projection_median"],
        "devonly_normal_projection_median": d["normal_projection_median"],
        "joint_Kseries_projection_median": j["Kseries_projection_median"],
        "devonly_Kseries_projection_median": d["Kseries_projection_median"],
    }

    cos_cols = [c for c in summary.columns if c.startswith("cos__")]
    for c in cos_cols:
        row[f"joint__{c}"] = j[c]
        row[f"devonly__{c}"] = d[c]
        row[f"delta_devonly_minus_joint__{c}"] = d[c] - j[c]

    paired_rows.append(row)

paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTDIR / "ae4_paired_seed_comparison.csv", index=False)
display(paired)

print("\nAcross-seed ranges:")
display(
    summary.groupby("scope")
    .agg(
        cancer_projection_min=("cancer_projection_global_fraction", "min"),
        cancer_projection_max=("cancer_projection_global_fraction", "max"),
        Kseries_projection_min=("Kseries_projection_median", "min"),
        Kseries_projection_max=("Kseries_projection_median", "max"),
    )
)


,seed,joint_cancer_projection,devonly_cancer_projection,devonly_minus_joint_cancer_projection,joint_normal_projection_median,devonly_normal_projection_median,joint_Kseries_projection_median,devonly_Kseries_projection_median,joint__cos__early__MPC_to_tip_acinar,devonly__cos__early__MPC_to_tip_acinar,...,delta_devonly_minus_joint__cos__total__MPC_to_tip_acinar,joint__cos__total__MPC_to_endocrine_islet,devonly__cos__total__MPC_to_endocrine_islet,delta_devonly_minus_joint__cos__total__MPC_to_endocrine_islet,joint__cos__total__endocrine_maturation,devonly__cos__total__endocrine_maturation,delta_devonly_minus_joint__cos__total__endocrine_maturation,joint__cos__total__biliary_maturation,devonly__cos__total__biliary_maturation,delta_devonly_minus_joint__cos__total__biliary_maturation
0,17,0.546546,0.952344,0.405798,0.573475,0.962256,0.540721,0.949034,-0.316056,-0.466671,...,-0.076216,0.051804,-0.218018,-0.269822,-0.179371,-0.265493,-0.086123,0.163298,0.079328,-0.083970
1,42,0.466932,0.957809,0.490876,0.515986,0.960001,0.452379,0.957891,-0.341200,-0.568177,...,-0.241851,0.031494,-0.198875,-0.230369,-0.113731,-0.200784,-0.087053,0.124616,0.242664,0.118048



Across-seed ranges:


,cancer_projection_min,cancer_projection_max,Kseries_projection_min,Kseries_projection_max
scope,,,,
development_only,0.952344,0.957809,0.949034,0.957891
joint,0.466932,0.546546,0.452379,0.540721


# AE4 closure

AE4 no longer trains anything. Its sole job is to determine how much the conclusions change when tumor cells are excluded from representation learning **after the audit corrections are held constant**.

The primary interpretation should separate:

1. absolute developmental reconstruction, which may be strongly representation-dependent;
2. the biologically revised malignant-direction alignments from AE3;
3. K-series-only residual organization and program association.

These are the quantities carried into AE5 and then compared with expression PCA/scVI in AE6.
